**NESTED VERSION**: the best epoch is chosen on 12 inner-validation patients drawn from the training folds (seed 42 + fold); the outer fold is used only for the final evaluation. Everything else is unchanged.

# Learned 3D landmark extractor: does reconstruction carry enough information?

Completes the error decomposition with a 2x2 comparison:
**{hand-crafted rule, learned 3D extractor} x {true mask, reconstructed mask}**.

A small 3D U-Net learns to find the expert hip and knee centres **on the true femur masks** (same folds, heatmap
targets). Mask-noise augmentation (smoothing/threshold, erosion/dilation, small bites and bumps) plus the same pose
augmentation as configuration C prepare it for imperfect reconstructions. It is then applied **unchanged** to the
predicted masks of AttentionUNet C and SwinUNETR C, on HFValid (fold that held each patient out) and VSD (5-fold ensemble).

- learned extractor on reconstructions reaches ~0.46 deg -> the hand-crafted rule was the limit; reconstruction carries the information
- stays near 0.53 deg -> the reconstructions themselves limit accuracy (mainly the hip)

**Inputs**: `preprocessed`, `run1` (split.json), `v3` output (v3_hfvalid_cv.csv, ckpt/attnC_*, swinC_*), VSD output,
optional: `lm2d` output (lm2d_hfvalid_cv.csv, lm2d_vsd.csv) for a direct comparison.
**Accelerator: GPU T4.** About 2.5-3.5 h. Output: `/kaggle/working/lm3d/` (+ lm3d_nested.zip).

In [ ]:
!pip install -q monai==1.6.1 einops

In [ ]:
import glob, json, re
from pathlib import Path
import numpy as np, pandas as pd, SimpleITK as sitk, torch
from scipy import ndimage
from scipy.stats import wilcoxon
from monai.networks.nets import AttentionUnet, SwinUNETR

def find(pat, required=True):
    hits = glob.glob(f"/kaggle/input/**/{pat}", recursive=True)
    assert hits or not required, f"{pat} not found under /kaggle/input: add the dataset that contains it"
    return Path(hits[0]).parent if hits else None
ROOT, SPLIT, RES, W = find("dataset.csv"), find("split.json"), find("v3_hfvalid_cv.csv"), find("attnC_fold0.pt")
VSDIN = find("prepare_log.csv", required=False)
FULL, VOX = (256, 128, 96), 2.0
device = "cuda" if torch.cuda.is_available() else "cpu"; amp = device == "cuda"
folds = json.load(open(SPLIT / "split.json")); ids = sorted(p for f in folds for p in f)
FOLD = {p: k for k, f in enumerate(folds) for p in f}
rd = lambda p: sitk.GetArrayFromImage(sitk.ReadImage(str(p)))
DRR = {p: [rd(ROOT / p / f"{v}.png").astype(np.float32)[::-1] / 65535.0 for v in ("ap", "lat")] for p in ids}
SEG = {p: rd(ROOT / p / "seg.nii.gz").astype(bool) for p in ids}
print(len(ids), "patients | VSD:", VSDIN, "| device", device)

In [ ]:
# extractor, offset, models: copied from train_v3
def surface_points(m):
    z, y, x = np.nonzero(m & ~ndimage.binary_erosion(m)); return np.stack([x, y, z], 1).astype(float)
def _sphere_lsq(p):
    s, *_ = np.linalg.lstsq(np.c_[2 * p, np.ones(len(p))], (p ** 2).sum(1), rcond=None)
    c = s[:3]; r2 = s[3] + c @ c; return (c, np.sqrt(r2)) if r2 > 0 else (None, None)
def fit_head(mask, prox=0.35, med=0.6, r_vox=(9, 14), n_iter=500, seed=0):
    p = surface_points(mask); z = p[:, 2]; p = p[z >= z.max() - prox * (z.max() - z.min())]
    xl, xh = p[:, 0].min(), p[:, 0].max(); p = p[p[:, 0] <= xl + med * (xh - xl)]
    rng = np.random.default_rng(seed); bn, bc, br = -1, None, None
    for _ in range(n_iter):
        c, r = _sphere_lsq(p[rng.choice(len(p), 4, replace=False)])
        if c is None or not (r_vox[0] <= r <= r_vox[1]): continue
        k = (np.abs(np.linalg.norm(p - c, axis=1) - r) < 2.0).sum()
        if k > bn: bn, bc, br = k, c, r
    for t in (2.0, 1.5, 1.0):
        inl = np.abs(np.linalg.norm(p - bc, axis=1) - br) < t; bc, br = _sphere_lsq(p[inl])
    return bc
def fit_knee(mask, dist_frac=0.30, win=2):
    z, y, x = np.nonzero(mask); sel = z <= z.min() + dist_frac * (z.max() - z.min())
    xs, ys, zs = x[sel], y[sel], z[sel]; zl = np.unique(zs)
    zc = zl[int(np.argmax([xs[zs == q].max() - xs[zs == q].min() for q in zl]))]
    m = np.abs(zs - zc) <= win
    return np.array([(xs[m].min() + xs[m].max()) / 2.0, ys[m].mean(), zc], float)
def drop_small(m, frac=0.05):
    lab, n = ndimage.label(m)
    if n <= 1: return m
    s = np.bincount(lab.ravel()); s[0] = 0; return np.isin(lab, np.nonzero(s >= frac * s.max())[0])
def expert(pid):
    lm = json.load(open(ROOT / pid / "landmarks.json"))["landmarks"]
    return np.array(lm["femurHead"]["voxel"]), np.array(lm["kneeCenter"]["voxel"])
KNEE_GT_RAW = {p: fit_knee(SEG[p]) for p in ids}
OFF = {f: np.mean([expert(p)[1] - KNEE_GT_RAW[p] for k, fl in enumerate(folds) if k != f for p in fl], 0) for f in range(5)}
OFF_ALL = np.mean([expert(p)[1] - KNEE_GT_RAW[p] for p in ids], 0)

def _remap(k):
    k = k.replace("module.", ""); k = re.sub(r"\bsub0\b", "submodule.0", k); k = re.sub(r"\bsub1\b", "submodule.1", k)
    return re.sub(r"\bsubconv\b", "submodule.conv", k)
make = {"attnC": lambda: AttentionUnet(spatial_dims=3, in_channels=2, out_channels=1, channels=(16, 32, 64, 128), strides=(2, 2, 2)),
        "swinC": lambda: SwinUNETR(in_channels=2, out_channels=1, feature_size=48, use_checkpoint=True, spatial_dims=3)}
NETS = {}
for k in make:
    for f in range(5):
        n = make[k](); n.load_state_dict({_remap(a): b for a, b in torch.load(W / f"{k}_fold{f}.pt", map_location=device).items()})
        NETS[(k, f)] = n.to(device).eval()
norm = lambda im: ((im - im.mean()) / (im.std() + 1e-6)).astype(np.float32)
def predict(k, ap, lat, fold=None):
    ap, lat = norm(ap), norm(lat)
    x = torch.from_numpy(np.stack([np.broadcast_to(ap[:, None, :], FULL), np.broadcast_to(lat[:, :, None], FULL)]).astype(np.float32))[None].to(device)
    with torch.no_grad(), torch.autocast(device_type="cuda", enabled=amp):
        nets = [NETS[(k, fold)]] if fold is not None else [NETS[(k, f)] for f in range(5)]
        p = torch.stack([torch.sigmoid(n(x)) for n in nets]).mean(0)
    return drop_small((p > 0.5)[0, 0].float().cpu().numpy() > 0)
print("models loaded:", len(NETS))

In [ ]:
# 1) predicted masks of the 3D models (HFValid: held-out fold model; VSD: 5-fold ensemble), then free the GPU
import gc, time, shutil
OUT = Path("/kaggle/working/lm3d_nested"); OUT.mkdir(parents=True, exist_ok=True)
LM2D = find("lm2d_hfvalid_cv.csv", required=False)
PRED = {k: {p: predict(k, *DRR[p], fold=FOLD[p]) for p in ids} for k in make}
def fill_solid(seg):
    s = ndimage.binary_fill_holes(seg); return np.stack([ndimage.binary_fill_holes(sl) for sl in s])
VSDC = {}
if VSDIN is not None:
    VD = VSDIN / "data"
    for fid in sorted(p.name for p in VD.iterdir() if (p / "landmarks.json").exists() and (p / "data.npz").exists()):
        d = np.load(VD / fid / "data.npz"); lm = json.load(open(VD / fid / "landmarks.json"))
        VSDC[fid] = dict(true=fill_solid(d["seg"]), hip=np.array(lm["hip"], float), knee=np.array(lm["knee"], float),
                         **{k: predict(k, d["ap"], d["lat"]) for k in make})
del NETS; gc.collect(); torch.cuda.empty_cache()
print("predicted masks:", {k: len(v) for k, v in PRED.items()}, "| VSD femurs with experts:", len(VSDC))

In [ ]:
# 2) learned 3D extractor: data, augmentation, model
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import matplotlib.pyplot as plt
ang3 = lambda a, b: float(np.degrees(np.arccos(np.clip(a @ b / np.linalg.norm(a) / np.linalg.norm(b), -1, 1))))
cor = lambda v: float(np.degrees(np.arctan2(-v[0], v[2]))); sag = lambda v: float(np.degrees(np.arctan2(v[1], v[2])))
from monai.networks.nets import UNet
def rot(rx, ry, rz):
    a, b, c = np.radians([rx, ry, rz])
    Rx = np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])
    Ry = np.array([[np.cos(b), 0, np.sin(b)], [0, 1, 0], [-np.sin(b), 0, np.cos(b)]])
    Rz = np.array([[np.cos(c), -np.sin(c), 0], [np.sin(c), np.cos(c), 0], [0, 0, 1]])
    return Rz @ Ry @ Rx
def pose_mask(m, lms, rng):                        # same pose augmentation as configuration C, on the mask
    R = rot(rng.uniform(-15, 15), rng.uniform(-10, 10), rng.uniform(-15, 15))
    P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]]); M = (P @ R @ P).T
    c = (np.array(FULL) - 1) / 2; t = rng.uniform(-4, 4, 3); off = c - M @ (c + t)
    mm = ndimage.affine_transform(m.astype(np.float32), M, off, order=1, cval=0.0) >= 0.5
    return mm, [(M.T @ (l[::-1] - off))[::-1] for l in lms]
def corrupt(m, rng):                               # imitate imperfect reconstructions
    if rng.random() < 0.5: m = ndimage.gaussian_filter(m.astype(np.float32), rng.uniform(0.5, 1.5)) > rng.uniform(0.35, 0.65)
    u = rng.random()
    if u < 0.25: m = ndimage.binary_erosion(m)
    elif u < 0.5: m = ndimage.binary_dilation(m)
    if rng.random() < 0.4:
        z, y, x = np.nonzero(m & ~ndimage.binary_erosion(m))
        for _ in range(rng.integers(1, 5)):
            i = rng.integers(len(z)); c = (z[i], y[i], x[i]); r = int(rng.integers(2, 6))
            sl = tuple(slice(max(ci - r, 0), min(ci + r + 1, n)) for ci, n in zip(c, m.shape))
            g = np.ogrid[sl]; d = sum((gi - ci) ** 2 for gi, ci in zip(g, c))
            m[sl][d <= r * r] = rng.random() < 0.5          # bump or bite
    return m
def g1(n, mu, s=2.0): return np.exp(-(np.arange(n) - mu) ** 2 / (2 * s * s)).astype(np.float32)
def heat3(lms):                                    # separable 3D Gaussians, landmarks given as voxel (x, y, z)
    return np.stack([g1(FULL[0], l[2])[:, None, None] * g1(FULL[1], l[1])[None, :, None] * g1(FULL[2], l[0])[None, None, :] for l in lms])
inside = lambda l: 0 <= l[0] < FULL[2] and 0 <= l[1] < FULL[1] and 0 <= l[2] < FULL[0]
class Train(Dataset):
    def __init__(self, ids_): self.ids = ids_
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        rng = np.random.default_rng(); pid = self.ids[i]; m, lms = SEG[pid], list(expert(pid))
        if rng.random() < 0.8:
            m2, l2 = pose_mask(m, lms, rng)
            if all(inside(l) for l in l2): m, lms = m2, l2
        m = corrupt(m.copy(), rng)
        return torch.from_numpy(m[None].astype(np.float32)), torch.from_numpy(heat3(lms))
make_lm = lambda: UNet(spatial_dims=3, in_channels=1, out_channels=2, channels=(8, 16, 32, 64, 128), strides=(2, 2, 2, 2), num_res_units=1).to(device)
def hm_loss(logits, target):
    B, C = logits.shape[:2]
    lp = F.log_softmax(logits.float().reshape(B, C, -1), -1); t = target.reshape(B, C, -1)
    return -((t / t.sum(-1, keepdim=True)) * lp).sum(-1).mean()
def decode3(prob, r=3):                            # argmax + probability-weighted centroid -> voxel (x, y, z)
    out = []
    for p in prob:
        c = np.unravel_index(p.argmax(), p.shape)
        sl = tuple(slice(max(ci - r, 0), min(ci + r + 1, n)) for ci, n in zip(c, p.shape))
        w = p[sl]; g = np.mgrid[sl]
        out.append(np.array([(w * g[k]).sum() / w.sum() for k in (2, 1, 0)]))
    return out
def lm3d(nets, mask):
    x = torch.from_numpy(mask[None, None].astype(np.float32)).to(device)
    with torch.no_grad(), torch.autocast(device_type="cuda", enabled=amp):
        ps = [F.softmax(n(x).float().flatten(2), -1).reshape(1, 2, *FULL) for n in nets]
    return decode3(torch.stack(ps).mean(0)[0].cpu().numpy())
# check: one augmented training mask with its target landmarks (max projections)
mk, hm = Train(ids[:1])[0]
fig, ax = plt.subplots(1, 2, figsize=(6, 6))
for a, axis in zip(ax, (1, 2)):
    a.imshow(mk[0].numpy().max(axis), cmap="gray", origin="lower"); a.contour(hm.numpy().sum(0).max(axis), levels=[0.5], colors=["r"])
ax[0].set_title("AP view: mask + targets"); ax[1].set_title("LAT view"); plt.show()

In [ ]:
# 3) train one extractor per fold on the true masks; best epoch = lowest mean landmark error on inner-validation TRUE masks (outer fold untouched)
EPOCHS, PATIENCE = 80, 15
for f in range(5):
    if (OUT / f"lm3d_fold{f}.pt").exists(): print("fold", f, "done"); continue
    torch.manual_seed(42 + f); t0 = time.time()
    tr = [p for k, fl in enumerate(folds) if k != f for p in fl]
    va = sorted(np.random.default_rng(42 + f).choice(tr, 12, replace=False).tolist())   # inner validation, from the training folds only
    tr = [p for p in tr if p not in va]; print(f"fold {f}: train {len(tr)}, inner val {len(va)}, outer test {len(folds[f])} (never used for selection)")
    net = make_lm(); opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS); scaler = torch.amp.GradScaler(enabled=amp)
    dl = DataLoader(Train(tr), batch_size=2, shuffle=True, num_workers=4, persistent_workers=True, drop_last=True)
    best, bad = np.inf, 0
    for ep in range(EPOCHS):
        net.train(); tl = 0
        for m, h in dl:
            with torch.autocast(device_type="cuda", enabled=amp): logits = net(m.to(device))
            loss = hm_loss(logits, h.to(device))
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tl += loss.item()
        sched.step(); net.eval()
        err = []
        for p in va:
            h_, k_ = lm3d([net], SEG[p]); eh, ek = expert(p)
            err.append((np.linalg.norm(h_ - eh) + np.linalg.norm(k_ - ek)) * VOX / 2)
        v = float(np.mean(err))
        if v < best: best, bad = v, 0; torch.save(net.state_dict(), OUT / f"lm3d_fold{f}.pt")
        else: bad += 1
        if ep % 10 == 0 or bad == 0: print(f"fold {f} ep {ep+1:3d} loss {tl/len(dl):.3f} val (true masks) {v:.2f} mm (best {best:.2f})", flush=True)
        if bad >= PATIENCE: break
    print(f"fold {f} finished: best {best:.2f} mm, {(time.time()-t0)/60:.0f} min", flush=True)

In [ ]:
# 4) the 2x2 comparison on HFValid and VSD
LMN = {}
for f in range(5):
    n = make_lm(); n.load_state_dict(torch.load(OUT / f"lm3d_fold{f}.pt", map_location=device)); LMN[f] = n.eval()
def errs(h, k, eh, ek):
    v, ve = h - k, eh - ek
    return dict(hip_mm=np.linalg.norm(h - eh) * VOX, knee_mm=np.linalg.norm(k - ek) * VOX, angle_deg=ang3(v, ve),
                coronal=abs(cor(v) - cor(ve)), sagittal=abs(sag(v) - sag(ve)))
def rule(mask, off):
    try: return fit_head(mask), fit_knee(mask) + off
    except Exception: return None
rows = []
for p in ids:
    eh, ek = expert(p); f = FOLD[p]
    for src, m in [("true", SEG[p])] + [(k, PRED[k][p]) for k in make]:
        r = rule(m, OFF[f])
        if r is not None: rows.append(dict(cohort="HFValid", case=p, source=src, extractor="rule", **errs(*r, eh, ek)))
        rows.append(dict(cohort="HFValid", case=p, source=src, extractor="learned", **errs(*lm3d([LMN[f]], m), eh, ek)))
for fid, d in VSDC.items():
    for src in ["true"] + list(make):
        r = rule(d[src], OFF_ALL)
        if r is not None: rows.append(dict(cohort="VSD", case=fid, source=src, extractor="rule", **errs(*r, d["hip"], d["knee"])))
        rows.append(dict(cohort="VSD", case=fid, source=src, extractor="learned", **errs(*lm3d(list(LMN.values()), d[src]), d["hip"], d["knee"])))
X = pd.DataFrame(rows); X["fail"] = X.hip_mm > 20; X.to_csv(OUT / "lm3d_results.csv", index=False)

q = lambda s: f"{s.median():.2f} [{s.quantile(.25):.2f}, {s.quantile(.75):.2f}]"
for coh in X.cohort.unique():
    print(f"\n{coh}: median [IQR]")
    for src in ["true"] + list(make):
        for ex in ["rule", "learned"]:
            g = X[(X.cohort == coh) & (X.source == src) & (X.extractor == ex)]
            print(f"  {src:6} {ex:8} axis {q(g.angle_deg)}  hip {q(g.hip_mm)}  knee {q(g.knee_mm)}  coronal {q(g.coronal)}"
                  f"  <2 deg {(g.angle_deg < 2).sum()}/{len(g)}  failures {int(g.fail.sum())}")
    if LM2D is not None:
        L = pd.read_csv(LM2D / ("lm2d_hfvalid_cv.csv" if coh == "HFValid" else "lm2d_vsd.csv")).dropna(subset=["angle_deg"])
        print(f"  2D baseline       axis {q(L.angle_deg)}  hip {q(L.hip_mm)}  knee {q(L.knee_mm)}  (n={len(L)})")

def paired(coh, a, b, metrics=("angle_deg", "hip_mm", "knee_mm")):
    A = X[(X.cohort == coh) & (X.source == a[0]) & (X.extractor == a[1])].set_index("case")
    B = X[(X.cohort == coh) & (X.source == b[0]) & (X.extractor == b[1])].set_index("case")
    print(f"  {coh}: {a} - {b}")
    for m in metrics:
        d = (A[m] - B[m].reindex(A.index)).dropna(); bs = [np.median(d.values[rng.integers(0, len(d), len(d))]) for _ in range(10000)]
        print(f"     {m:10} median diff {np.median(d):+.3f}  95% CI [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]  p {wilcoxon(d).pvalue:.3g}  (n={len(d)})")
rng = np.random.default_rng(0)
print("\npaired comparisons (Wilcoxon, unadjusted; bootstrap CI of median paired difference)")
for coh in X.cohort.unique():
    for k in make:
        paired(coh, (k, "learned"), (k, "rule"))
    paired(coh, ("true", "learned"), ("true", "rule"))
    if LM2D is not None:
        L = pd.read_csv(LM2D / ("lm2d_hfvalid_cv.csv" if coh == "HFValid" else "lm2d_vsd.csv")).rename(columns={"patient": "case", "femur": "case"})
        L = L.dropna(subset=["angle_deg"]).set_index("case")
        for k in make:
            A = X[(X.cohort == coh) & (X.source == k) & (X.extractor == "learned")].set_index("case")
            d = (A.angle_deg - L.angle_deg.reindex(A.index)).dropna()
            print(f"  {coh}: {k} learned - 2D baseline, axis median diff {np.median(d):+.3f}  p {wilcoxon(d).pvalue:.3g}  (n={len(d)})")
shutil.make_archive("/kaggle/working/lm3d_nested", "zip", OUT); print("\ndone -> download lm3d_nested.zip")